# 対話の履歴を管理して、長くなったら古いターンを捨てる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/21_dialogue/chat_history.ipynb)

NLP クックブック 21-05 ／ 対話システム

LLM と何ターンも対話を続けたい。履歴が長くなると入力が膨らんで遅くなり、やがて上限を超えるので、長さを管理したい。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"   # 3B の日本語モデル。CPU でも 1 応答あたり数秒〜十数秒 (メモリ約 7GB)
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

class Chat:
    def __init__(self, system, max_input_tokens=200):
        self.system = {"role": "system", "content": system}
        self.history = []                       # user と assistant の発話を交互に積む
        self.max_input_tokens = max_input_tokens

    def n_tokens(self, messages):
        return len(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True)["input_ids"])

    def window(self):
        """システムプロンプトは必ず残し、上限に収まるまで古いターン (user と assistant の組) を捨てる"""
        history = list(self.history)
        while len(history) > 1 and self.n_tokens([self.system] + history) > self.max_input_tokens:
            history = history[2:]
        return [self.system] + history

    def say(self, text, max_new_tokens=60):
        self.history.append({"role": "user", "content": text})
        messages = self.window()
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        with torch.no_grad():
            out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        self.history.append({"role": "assistant", "content": reply})
        return reply, inputs["input_ids"].shape[1], (len(messages) - 1) // 2

chat = Chat("あなたは旅行の相談に乗るアシスタントです。1〜2 文で短く答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "秋におすすめのお寺を 1 つ教えてください。", "そこへは京都駅からどう行けばいいですか？",
             "近くでお昼を食べるなら何がいいですか？", "ところで、私の名前を覚えていますか？"]:
    reply, n_in, n_turns = chat.say(user)
    print(f"あなた: {user}\nボット: {reply}\n   (入力 {n_in} トークン、直近 {n_turns} ターン分を使用)")

- LLM は状態を持たない。「覚えている」ように見えるのは、毎回これまでの対話をすべて入力しているから。
- 古いターンを捨てると、そこで話したこと (最初に名乗った名前) は忘れる。上限を小さくしてあるこの例では、最後の質問でそれが起きる。
- 捨てるときは user と assistant の組で捨てる。片方だけ残すと、チャットテンプレートの交互の並びが崩れる。
- 大事な情報を失わないための方法が、次項の「要約して覚えておく」。